# Exploratory data analysis: UCI Default of Credit Card Clients

Exploration only. **Nothing in the pipeline imports this notebook.** It documents the data facts that the
validation schema (`src/data/validation.py`) and preprocessing (`src/features/preprocessing.py`) rely on.

Run from the repo root after `dvc pull -r dagshub` (needs `data/raw/credit_default.csv`).
Point `DATA` at the synthetic file (`python -m src.data.synthetic`) only for a dry run: numbers are then not real.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA = Path("data/raw/credit_default.csv")   # real data; dry run: data/smoke/raw/credit_default.csv
TARGET, ID = "default", "customer_id"
df = pd.read_csv(DATA)
PAY = [f"PAY_{i}" for i in range(1, 7)]
BILL = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYAMT = [f"PAY_AMT{i}" for i in range(1, 7)]
print(DATA, df.shape)
df.head()

## 1. Shape, types, missing values, duplicates

In [ ]:
print(df.dtypes.value_counts())
print("nulls:", int(df.isna().sum().sum()))
print("duplicate IDs:", int(df[ID].duplicated().sum()))
print("duplicate rows ignoring ID:", int(df.drop(columns=[ID]).duplicated().sum()))
df.describe().T

## 2. Class balance (the target is imbalanced, which is why the models use class weights)

In [ ]:
rate = df[TARGET].mean()
print(f"default rate: {rate:.2%}  (positives {int(df[TARGET].sum()):,} of {len(df):,})")
df[TARGET].value_counts().plot.bar(title="Target counts (0 = paid, 1 = default)", rot=0)
plt.show()

## 3. Categorical columns and the undocumented codes
The UCI documentation lists `EDUCATION` 1-4 and `MARRIAGE` 1-3, but the file also contains `EDUCATION` 0/5/6 and `MARRIAGE` 0.
They are kept (the schema allows them) and handled in preprocessing. This section shows how many rows are affected
and whether they behave differently.

In [ ]:
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    t = df.groupby(col)[TARGET].agg(rows="size", default_rate="mean")
    t["share"] = t["rows"] / len(df)
    print(col); display(t.round(4)); print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["SEX", "EDUCATION", "MARRIAGE"]):
    df.groupby(col)[TARGET].mean().plot.bar(ax=ax, rot=0, title=f"Default rate by {col}")
plt.tight_layout(); plt.show()

## 4. `PAY_*` coding
`PAY_k` is the repayment status in month k: **-2** no consumption, **-1** paid in full, **0** revolving credit / minimum paid,
**1..8** months of delay. It is a *status code*, not a plain ordinal amount, so the engineered features
(max/mean delay, count of delinquent months) treat values <= 0 as "not late".

In [ ]:
all_pay = pd.concat([df[c] for c in PAY])
print(all_pay.value_counts().sort_index())
print("share of PAY values > 0 (late):", round(float((all_pay > 0).mean()), 4))
df.groupby("PAY_1")[TARGET].agg(rows="size", default_rate="mean").round(3)

In [ ]:
ax = df.groupby("PAY_1")[TARGET].mean().plot.bar(rot=0, title="Default rate by most recent payment status (PAY_1)")
ax.set_ylabel("default rate"); plt.show()

## 5. Numeric ranges and skew (what the validation ranges and scaling must cope with)

In [ ]:
num = ["LIMIT_BAL", "AGE"] + BILL + PAYAMT
df[num].agg(["min", "median", "mean", "max", "skew"]).T

In [ ]:
print("negative bills:", {c: int((df[c] < 0).sum()) for c in BILL if (df[c] < 0).any()})
print("non-positive limit rows:", int((df["LIMIT_BAL"] <= 0).sum()))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["LIMIT_BAL", "AGE", "BILL_AMT1"]):
    df[col].plot.hist(bins=50, ax=ax, title=col)
plt.tight_layout(); plt.show()

## 6. Which raw features relate to the target (sanity check, not feature selection)

In [ ]:
corr = df.drop(columns=[ID]).corr(numeric_only=True)[TARGET].drop(TARGET)
corr.sort_values(key=abs, ascending=False).head(12).round(3)

In [ ]:
df["utilisation1"] = (df["BILL_AMT1"] / df["LIMIT_BAL"]).clip(-1, 5)
df["max_delay"] = df[PAY].clip(lower=0).max(axis=1)
df["n_late"] = (df[PAY] > 0).sum(axis=1)
print(df[["utilisation1", "max_delay", "n_late", TARGET]].corr()[TARGET].round(3))
df.groupby("n_late")[TARGET].agg(rows="size", default_rate="mean").round(3)

## 7. Sensitive attributes
`SEX`, `MARRIAGE`, `AGE` are **not model inputs** (`features.exclude_sensitive_from_model: true`); they are used only to audit
outcomes (`docs/fairness.md`). The default rate by age band below is the base-rate context for that audit.

In [ ]:
bands = pd.cut(df["AGE"], [0, 29, 39, 49, 59, 200], labels=["<30", "30-39", "40-49", "50-59", "60+"])
df.groupby(bands, observed=True)[TARGET].agg(rows="size", default_rate="mean").round(3)

## 8. Findings to carry into the pipeline
Fill in after running on the real data:
- Default rate and the imbalance handling it implies.
- Rows with undocumented `EDUCATION`/`MARRIAGE` codes: count, default rate, how preprocessing treats them.
- `PAY_*` coding and why delay features ignore codes <= 0.
- Skewed amounts (bills/payments) and negative bills; scaling choice.
- Duplicate rows ignoring ID (kept) and why.